In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
print("Libraries successfully imported!")

In [ ]:
#Load engineered dataset
matches= pd.read_csv("../data/processed/epl_matches_features.csv")
matches["Date"]=pd.to_datetime(matches["Date"])
matches=matches.sort_values("Date").reset_index(drop=True)
print("Dataset Shape: ", matches.shape)
matches.head()

In [ ]:
#Target:
y=matches["MatchResult"]
y.value_counts()

In [ ]:
#Input Features
feature_columns = [
    "HomeRecentPoints",
    "AwayRecentPoints",
    "HomeRecentWinRate",
    "AwayRecentWinRate",
    "HomeRecentGoalsScored",
    "AwayRecentGoalsScored",
    "HomeRecentGoalsConceded",
    "AwayRecentGoalsConceded",
    "HomeRecentGoalDifference",
    "AwayRecentGoalDifference",
    "HomeMatchesPlayedBefore",
    "AwayMatchesPlayedBefore",
    "HomeDaysSinceLastMatch",
    "AwayDaysSinceLastMatch"
]

X = matches[feature_columns].copy()
print("Input features shape:", X.shape)
print("Target shape:", y.shape)
X.head()
missing_values=X.isnull().sum()
print(missing_values[missing_values>0])

In [ ]:
#Sort chronologically
matches=matches.sort_values("Date").reset_index(drop=True)
#Feature and Target in chronological order
X=matches[feature_columns].copy()
y=matches["MatchResult"].copy()

#Train and Test
split=int(len(matches)*0.80)
X_train=X.iloc[:split].copy()
X_test=X.iloc[split:].copy()

y_train=y.iloc[:split].copy()
y_test=y.iloc[split:].copy()
print("Train data: ", X_train)
print("Test data: ", X_test)
print("Training date range:",matches["Date"].iloc[0],"to",matches["Date"].iloc[split- 1])


In [ ]:
numeric_pipeline=Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
X_train_processed=numeric_pipeline.fit_transform(X_train)
X_test_processed=numeric_pipeline.transform(X_test)
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

In [ ]:
label_encoder=LabelEncoder()
y_train_encoded=label_encoder.fit_transform(y_train)
y_test_encoded=label_encoder.transform(y_test)
print("Classes: ", label_encoder.classes_)
y_train_encoded[:10]

In [ ]:
print("Training Samples: ", len(X_train))
print("Testing Samples: ", len(X_test))
print("Training class distribution: ", y_train.value_counts())
print("Testing class distribution: ", y_test.value_counts())
print("Missing values after preprocessing:")
print(np.isnan(X_train_processed).sum())
print(np.isnan(X_test_processed).sum())

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)
print("Training and testing dataset saved.")